# 🧹 Notebook 02 — Data Preprocessing
**Goal:** Clean, deduplicate, resize and split the extracted frames.

| Step | Action |
|---|---|
| 1 | Load dataset, count images |
| 2 | Remove corrupted images |
| 3 | Remove exact duplicates (MD5 hash) |
| 4 | Resize all images to 224×224 |
| 5 | Split → Train 80% / Val 10% / Test 10% |
| 6 | Save final dataset to Google Drive |

➡ **Input:** `dataset_frames/`  ➡ **Output:** `dataset_final/`

In [ ]:
# ════════════════════════════════════════════════════════════════════
#  STEP 0 — Imports & Drive Mount
# ════════════════════════════════════════════════════════════════════

import os, cv2, hashlib, shutil, random
from pathlib import Path
from google.colab import drive
import matplotlib.pyplot as plt

drive.mount('/content/drive')

# ╔══════════════════════════════════════════════════════════╗
# ║          PROJECT-WIDE CONSTANTS — do not change          ║
# ║  These must be identical across all notebooks & scripts  ║
# ╚══════════════════════════════════════════════════════════╝
DATASET_PATH   = '/content/drive/MyDrive/dataset_final'   # split dataset
FRAMES_PATH    = '/content/drive/MyDrive/dataset_frames'  # raw frames
CLASS_NAMES    = ['cheating', 'not_cheating']             # alphabetical (ImageFolder order)
IMG_SIZE       = (224, 224)
BATCH_SIZE     = 32
NUM_EPOCHS     = 20
PATIENCE       = 5

# ── Best model (chosen from Notebook 04 results) ─────────────
# Custom CNN: Acc=0.9780, F1=0.9780, ROC-AUC=0.9995, Params=~200K
# Lightest AND best ROC-AUC → perfect for real-time inference
BEST_MODEL_NAME = 'Custom CNN'
BEST_MODEL_FILE = 'cnn_cheating_model.pth'
BEST_MODEL_KEY  = 'cnn'

# Normalisation for Custom CNN (trained from scratch — NOT ImageNet)
CLF_MEAN = [0.5, 0.5, 0.5]
CLF_STD  = [0.5, 0.5, 0.5]

# All model .pth filenames (for reference / comparison notebook)
MODEL_FILES = {
    'cnn':          'cnn_cheating_model.pth',
    'resnet18':     'resnet18_cheating_model.pth',
    'efficientnet': 'efficientnet_cheating.pth',
    'vit':          'Vision_Transformer.pth',
    'mobilenet':    'mobilenetv2_model.pth',
}

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
print(f"📂 Input  (raw frames) : {FRAMES_PATH}")
print(f"📂 Output (split data) : {DATASET_PATH}")

# Auto-detect classes
CLASS_NAMES_FOUND = sorted([
    d for d in os.listdir(FRAMES_PATH)
    if os.path.isdir(os.path.join(FRAMES_PATH, d))
])
print(f"📂 Classes found: {CLASS_NAMES_FOUND}")


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  STEP 1 — Dataset Overview
# ════════════════════════════════════════════════════════════════════

print("\n📊 Dataset Overview:\n")
total = 0
for cls in CLASS_NAMES_FOUND:
    cls_path = os.path.join(FRAMES_PATH, cls)
    images   = [f for f in os.listdir(cls_path) if Path(f).suffix.lower() in IMG_EXTS]
    print(f"  📁 {cls:<18} → {len(images):>5} images")
    total += len(images)
print(f"\n  {'TOTAL':<18} → {total:>5} images")


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  STEP 2 — Remove Corrupted Images
# ════════════════════════════════════════════════════════════════════

removed = 0
for cls in CLASS_NAMES_FOUND:
    cls_path = os.path.join(FRAMES_PATH, cls)
    for img_name in os.listdir(cls_path):
        img_path = os.path.join(cls_path, img_name)
        try:
            img = cv2.imread(img_path)
            if img is None:
                os.remove(img_path)
                removed += 1
        except Exception:
            os.remove(img_path)
            removed += 1
print(f"🧹  Removed corrupted images: {removed}")


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  STEP 3 — Remove Exact Duplicates (MD5 hash)
# ════════════════════════════════════════════════════════════════════

hashes     = {}
duplicates = 0

for cls in CLASS_NAMES_FOUND:
    cls_path = os.path.join(FRAMES_PATH, cls)
    for img_name in os.listdir(cls_path):
        img_path = os.path.join(cls_path, img_name)
        try:
            with open(img_path, 'rb') as f:
                h = hashlib.md5(f.read()).hexdigest()
            if h in hashes:
                os.remove(img_path)
                duplicates += 1
            else:
                hashes[h] = img_path
        except Exception:
            continue

print(f"🗑️   Removed duplicate images: {duplicates}")


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  STEP 4 — Resize All Images to 224×224
# ════════════════════════════════════════════════════════════════════

resized = 0
for cls in CLASS_NAMES_FOUND:
    cls_path = os.path.join(FRAMES_PATH, cls)
    for img_name in os.listdir(cls_path):
        if Path(img_name).suffix.lower() not in IMG_EXTS:
            continue
        img_path = os.path.join(cls_path, img_name)
        try:
            img = cv2.imread(img_path)
            if img is None:
                continue
            img = cv2.resize(img, IMG_SIZE)
            cv2.imwrite(img_path, img)
            resized += 1
        except Exception:
            continue

print(f"🖼️   Resized {resized} images to {IMG_SIZE[0]}×{IMG_SIZE[1]}")


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  STEP 5 — Train / Val / Test Split  (80 / 10 / 10)
# ════════════════════════════════════════════════════════════════════

FINAL_DIR   = '/content/dataset_final'
TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1

shutil.rmtree(FINAL_DIR, ignore_errors=True)
for split in ['train', 'val', 'test']:
    for cls in CLASS_NAMES_FOUND:
        os.makedirs(os.path.join(FINAL_DIR, split, cls), exist_ok=True)

random.seed(42)   # reproducible split

for cls in CLASS_NAMES_FOUND:
    cls_path = os.path.join(FRAMES_PATH, cls)
    images   = sorted([
        f for f in os.listdir(cls_path)
        if Path(f).suffix.lower() in IMG_EXTS
    ])
    random.shuffle(images)

    n          = len(images)
    train_end  = int(n * TRAIN_RATIO)
    val_end    = int(n * (TRAIN_RATIO + VAL_RATIO))

    splits_map = {
        'train': images[:train_end],
        'val':   images[train_end:val_end],
        'test':  images[val_end:],
    }

    for split, imgs in splits_map.items():
        for img in imgs:
            src = os.path.join(cls_path, img)
            dst = os.path.join(FINAL_DIR, split, cls, img)
            shutil.copy2(src, dst)

    print(f"\n  📂 {cls}")
    print(f"     Train : {len(splits_map['train'])}")
    print(f"     Val   : {len(splits_map['val'])}")
    print(f"     Test  : {len(splits_map['test'])}")

print("\n✅  Split complete.")


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  STEP 6 — Class Balance Plot & Save to Google Drive
# ════════════════════════════════════════════════════════════════════

# Plot
splits = ['train', 'val', 'test']
x      = range(len(CLASS_NAMES_FOUND))
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, split in zip(axes, splits):
    counts = [len(os.listdir(os.path.join(FINAL_DIR, split, cls)))
              for cls in CLASS_NAMES_FOUND]
    bars = ax.bar(CLASS_NAMES_FOUND, counts, color=['#E74C3C', '#2ECC71'])
    for bar, c in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(c), ha='center', fontweight='bold')
    ax.set_title(split.capitalize())
    ax.set_ylabel('Images')
    ax.set_ylim(0, max(counts) * 1.2)

plt.suptitle('Class Distribution per Split', fontweight='bold')
plt.tight_layout()
plt.show()

# Save to Drive
DRIVE_SAVE = DATASET_PATH
shutil.rmtree(DRIVE_SAVE, ignore_errors=True)
shutil.copytree(FINAL_DIR, DRIVE_SAVE)
print(f"💾  Dataset saved to Drive: {DRIVE_SAVE}")
print("✅  Preprocessing complete. Proceed to Notebook 03.x for training.")
